![SGSSS Logo](https://github.com/SGSSSonline/text-analysis-for-social-scientists/blob/main/img/SGSSS_Stacked.png?raw=true)

# Practical 3: Topic Modelling

Topic models are one of the most widely used unsupervised methods in computational text analysis. They allow researchers to discover the latent thematic structure of a corpus — that is, to identify the topics that documents are about — without requiring any prior labelling or classification.

In this practical, we work through the full topic modelling pipeline: from preprocessing and representation, through model fitting and interpretation, to evaluation and visualisation. We use **Latent Dirichlet Allocation (LDA)**, the foundational topic model introduced by Blei, Ng, and Jordan (2003), which remains a workhorse of applied text analysis in the social sciences.

## Aims

This practical has two aims:

1. **Demonstrate how to use R to fit, interpret, and evaluate topic models** on social science text data.
2. **Cultivate computational thinking skills** — understanding the choices involved in topic modelling and how they affect substantive conclusions.

### Lesson Details

| | |
| --- | --- |
| **Level** | Introductory |
| **Time** | ~75 minutes |
| **Pre-requisites** | Practical 1 (Preprocessing Text) |
| **Learning outcomes** | Understand the intuition behind Latent Dirichlet Allocation (LDA) |
| | Be able to prepare text data for topic modelling using quanteda |
| | Be able to fit and interpret an LDA model using the topicmodels package |
| | Be able to visualise topics using tidytext and ggplot2 |
| | Be able to examine document-topic assignments |
| | Understand how to evaluate topic models using perplexity |
| | Appreciate the effect of preprocessing choices on topic model outputs |

## Guide to Using This Resource

This is a **Jupyter Notebook** — an interactive document that combines text, code, and output in a single environment. If you are viewing this in **Google Colab**, you are running the notebook in the cloud, which means you do not need to install anything on your own machine.

**Note: This notebook uses R.** In Google Colab, you need to change the runtime: go to **Runtime > Change runtime type > select R**.

A notebook is made up of **cells**. There are two main types:

- **Markdown cells** contain formatted text (like this one). They provide explanations, instructions, and context.
- **Code cells** contain R code that you can execute. Code cells are displayed with a grey background and have a play button on the left.

To **run a cell**, click on it and press `Shift+Enter` (or click the play button). The output will appear directly below the cell. You should run the code cells **in order**, from top to bottom, as later cells often depend on variables or packages loaded in earlier cells.

If you are using **Google Colab**, make sure to save a copy of this notebook to your own Google Drive first: go to **File > Save a copy in Drive**.

If you are new to Jupyter Notebooks and would like a more detailed introduction, see the excellent materials by Dani Arribas-Bel: [https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb](https://github.com/darribas/gds19/blob/master/content/labs/lab_00.ipynb)

### Interactive Test

Let's make sure everything is working. Run the cell below and enter your name when prompted.

In [ ]:
cat("Hello! What is your name?\n")
name <- readline()
cat(paste0("Welcome to the course, ", name, "!\n"))

## Setup

Before we begin, we need to install and load the R packages we will use throughout this practical. These include:

- **quanteda** — for tokenisation, stopword removal, and constructing document-feature matrices.
- **topicmodels** — for fitting Latent Dirichlet Allocation (LDA) models (Grun and Hornik, 2011).
- **tidytext** — for extracting tidy data frames from topic models (Silge and Robinson, 2016).
- **ggplot2** — for creating publication-quality visualisations.
- **dplyr** — for data manipulation.
- **reshape2** — for reshaping data between wide and long formats.

In [ ]:
# Install packages (if needed on Colab)
install.packages(c("quanteda", "topicmodels", "tidytext", "ggplot2", "dplyr", "reshape2"), quiet = TRUE)

library(quanteda)
library(topicmodels)
library(tidytext)
library(ggplot2)
library(dplyr)
library(reshape2)

cat("Successfully loaded packages\n")

## Loading the Corpus

We will work with the same set of **synthetic** UK parliamentary speeches used in Practical 1. The speakers and speeches are fictional, designed to reflect realistic variation in policy topics and party positions.

> **A note on corpus size:** Our sample corpus is small (20 speeches). In practice, topic models work best with larger corpora (hundreds or thousands of documents). We use this small sample for learning purposes — the code scales to larger datasets. With a small corpus, topics may be less clearly separated and perplexity scores less reliable. Keep this in mind as you interpret the results.

In [ ]:
# Create a sample dataset of parliamentary speech excerpts
df <- data.frame(
  date = c(
    "2024-01-15", "2024-01-15", "2024-01-22", "2024-01-22", "2024-02-05",
    "2024-02-05", "2024-02-12", "2024-02-12", "2024-02-19", "2024-02-19",
    "2024-03-04", "2024-03-04", "2024-03-11", "2024-03-11", "2024-03-18",
    "2024-03-18", "2024-03-25", "2024-03-25", "2024-04-01", "2024-04-01"
  ),
  speaker = c(
    "Margaret Thornton", "James Whitfield", "Sarah Okonkwo", "David Hargreaves",
    "Emily Chen", "Robert MacLeod", "Fatima Hussain", "Thomas Brennan",
    "Catherine Lloyd", "Andrew Patel", "Helen Murray", "Kwame Asante",
    "Patricia Gallagher", "Ravi Sharma", "Fiona Blackwood", "Marcus Johnson",
    "Alison Crawford", "Yusuf Ali", "Diana Novak", "Christopher Reeves"
  ),
  party = c(
    "Labour", "Conservative", "Labour", "Conservative", "Labour",
    "SNP", "Labour", "Conservative", "Liberal Democrat", "Labour",
    "SNP", "Labour", "Conservative", "Labour", "SNP",
    "Conservative", "Labour", "Labour", "Liberal Democrat", "Conservative"
  ),
  speech_text = c(
    "The National Health Service remains the cornerstone of our social contract with the British people. We must invest in frontline services and address the chronic staffing shortages that are putting patients at risk. The waiting lists have grown to record levels and this government must take urgent action to reduce them.",
    "Economic growth is the foundation upon which all public services depend. By cutting taxes and reducing unnecessary regulation, we can unleash the potential of British businesses. The private sector, not the state, is the engine of prosperity and job creation in this country.",
    "The climate crisis demands immediate and ambitious action from this House. We cannot afford to delay investment in renewable energy infrastructure. Our children and grandchildren will judge us by the decisions we make today on carbon emissions and environmental protection.",
    "Immigration policy must be fair but firm. We need a points-based system that attracts the skilled workers our economy needs while maintaining control of our borders. The British people voted for sovereignty and we must deliver on that promise.",
    "Education is the great equaliser in our society. Every child, regardless of their background or postcode, deserves access to excellent teaching and modern facilities. We must close the attainment gap between the most and least disadvantaged pupils in our schools.",
    "Scotland's interests are consistently overlooked by this Westminster government. The devolution settlement is being undermined at every turn. The people of Scotland deserve the right to determine their own future and to have their voice heard on matters that affect their daily lives.",
    "The housing crisis is devastating communities across the country. Young people cannot afford to buy their first home and rents are consuming an ever larger share of household incomes. We need a massive programme of social housing construction to address this emergency.",
    "Defence spending must remain a top priority for this government. The threats we face from hostile state actors are growing more complex and more dangerous. We must ensure our armed forces have the equipment and resources they need to keep this nation safe.",
    "Mental health services are chronically underfunded and overstretched. Too many people are waiting months or even years for treatment they desperately need. Parity of esteem between physical and mental health must become a reality, not just a slogan.",
    "The cost of living crisis is hitting working families hardest. Energy bills have skyrocketed and food prices continue to rise at alarming rates. The government must do more to support those who are struggling to heat their homes and feed their children.",
    "The fishing communities of Scotland have been betrayed by broken promises on Brexit. Access to our waters and fair quota allocations are essential for the survival of these coastal communities. We will continue to fight for the rights of Scottish fishermen in this House.",
    "Public transport infrastructure is failing passengers across the north of England. Years of underinvestment have left communities isolated and reliant on expensive and unreliable services. We need a transport revolution that connects people to jobs, education, and opportunity.",
    "Law and order must be restored to our streets. Violent crime and antisocial behaviour are blighting communities and the police need the resources and powers to tackle these problems effectively. We will always stand on the side of victims and law-abiding citizens.",
    "The arts and creative industries contribute billions to our economy and enrich the cultural life of this nation. Cuts to arts funding have devastated regional theatres, museums, and music venues. Investment in culture is not a luxury; it is an investment in our communities and our identity.",
    "Rural communities in Scotland face unique challenges that this government fails to understand. Broadband connectivity, access to healthcare, and affordable transport are not luxuries but necessities. The centralisation of services in urban areas is leaving rural Scotland behind.",
    "Free trade agreements are essential for our post-Brexit economic strategy. British goods and services must have access to global markets on the most favourable terms possible. We are building new trading relationships that will benefit every region of the United Kingdom.",
    "Workers' rights must be strengthened, not eroded, in the modern economy. The rise of zero-hours contracts and the gig economy has created a new class of precarious employment. Every worker deserves fair pay, decent conditions, and the security of knowing their rights are protected by law.",
    "Child poverty is a stain on our national conscience. Over four million children in this country are growing up in poverty, and the numbers are rising. We must reform the welfare system to ensure that no child goes hungry and every family has a decent standard of living.",
    "Local government is the backbone of democracy in this country. Councils are being asked to do more with less, and essential services are being cut to the bone. Proper funding for local authorities is not just desirable; it is essential for the health of our democratic institutions.",
    "Science and innovation are the keys to our future prosperity. The United Kingdom has world-leading universities and research institutions that must be supported and celebrated. Government investment in research and development is an investment in the jobs and industries of tomorrow."
  ),
  stringsAsFactors = FALSE
)

cat(paste0("Dataset shape: ", nrow(df), " speeches, ", ncol(df), " columns\n"))
head(df)

## Preprocessing

Before fitting a topic model, we need to preprocess the text. This is a brief recap of the steps covered in Practical 1. We will:

1. **Create a quanteda corpus** from the speech text.
2. **Tokenise** each speech into individual words.
3. **Lowercase** all tokens.
4. **Remove punctuation** and numbers.
5. **Remove stopwords** — common words that carry little substantive meaning.

We build the full preprocessing pipeline using the `quanteda` package, producing a **Document-Feature Matrix (DFM)** — the standard input format for topic models in R.

In [ ]:
# Create a quanteda corpus
corp <- corpus(df, text_field = "speech_text")

# Build the DFM via the quanteda pipeline
my_dfm <- corp |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  tokens_remove(stopwords("en")) |>
  dfm()

cat(paste0("DFM dimensions: ", nrow(my_dfm), " documents x ", ncol(my_dfm), " features\n"))
cat(paste0("\nTop 15 features in the corpus:\n"))
topfeatures(my_dfm, n = 15)

Notice how the preprocessing has stripped away grammatical scaffolding — articles, prepositions, auxiliary verbs — and left us with the substantive content words. These are the words that will drive the topic model.

## Preparing for LDA

Latent Dirichlet Allocation (LDA) does not work directly with raw text. It requires a numerical document-term representation. We already have a DFM from `quanteda`, but the `topicmodels` package (Grun and Hornik, 2011) expects input in a slightly different format — a `DocumentTermMatrix` object from the `tm` package.

Fortunately, `quanteda` provides the `convert()` function to translate between formats seamlessly.

### Filtering the Vocabulary

Before converting, we trim the DFM to remove very rare and very common features. Words that appear in only one document are too rare to reveal shared themes. Words that appear in almost every document are too common to distinguish between topics:

- `min_docfreq = 2`: Remove words that appear in fewer than 2 documents.
- `max_docfreq = 0.8`: Remove words that appear in more than 80% of documents (expressed as a proportion).

This is analogous to the TF-IDF logic from Practical 1: we want words that are frequent enough to matter but distinctive enough to differentiate.

In [ ]:
# Trim the DFM: remove rare and overly common features
dfm_trimmed <- dfm_trim(my_dfm, min_docfreq = 2, max_docfreq = 0.8, docfreq_type = "prop")

cat(paste0("DFM before trimming: ", nrow(my_dfm), " docs x ", ncol(my_dfm), " features\n"))
cat(paste0("DFM after trimming:  ", nrow(dfm_trimmed), " docs x ", ncol(dfm_trimmed), " features\n"))

In [ ]:
# Convert the quanteda DFM to a topicmodels-compatible DocumentTermMatrix
dtm <- convert(dfm_trimmed, to = "topicmodels")

cat(paste0("DocumentTermMatrix dimensions: ", nrow(dtm), " documents x ", ncol(dtm), " terms\n"))
cat(paste0("Non-zero entries: ", length(dtm$v), "\n"))

We now have a `DocumentTermMatrix` object ready for the `topicmodels` package. Each row is a document, each column is a word, and each cell contains the count of how many times that word appears in that document.

## Fitting Your First LDA Model

We are now ready to fit a **Latent Dirichlet Allocation (LDA)** model. LDA is a generative probabilistic model that assumes each document is a mixture of topics, and each topic is a distribution over words. The model learns both distributions simultaneously from the data.

The key parameters are:

| Parameter | Description |
| --- | --- |
| `dtm` | The document-term matrix we created above |
| `k` | The number of topics to discover (K) — we start with 3 |
| `control` | A list of control parameters, including `seed` for reproducibility |

We choose **K=3** as a starting point. With only 20 documents, a small number of topics is appropriate — we can experiment with other values later.

Setting `seed = 42` ensures reproducibility — running the code again will produce the same results.

In [ ]:
# Fit an LDA model with 3 topics
lda_model <- LDA(dtm, k = 3, control = list(seed = 42))

cat("LDA model fitted successfully.\n")
cat(paste0("Number of topics: ", lda_model@k, "\n"))
cat(paste0("Vocabulary size: ", ncol(dtm), "\n"))
cat(paste0("Number of documents: ", nrow(dtm), "\n"))

## Interpreting Topics

The model has identified 3 topics. Each topic is a probability distribution over the vocabulary: some words have a high probability under a given topic (they are strongly associated with it) and others have a low probability.

Let's examine the top 10 words for each topic using the `terms()` function from the `topicmodels` package. This extracts the most probable words for each topic.

In [ ]:
# Display the top 10 words for each topic
top_terms <- terms(lda_model, 10)

cat("Top 10 words per topic:\n\n")
print(top_terms)

### Reading the Output

Each column represents a topic, and the rows list the most probable words in descending order. When interpreting topics, look for **coherent themes** — groups of words that relate to a common subject or concept.

For example, if a topic's top words include "health", "services", "waiting", you might label it as a **Health/NHS** topic. If another includes "economy", "trade", "british", you might call it an **Economy** topic.

Topic labelling is a qualitative, interpretive step. There is no algorithm that names topics for you — the researcher must read the top words and assign a meaningful label based on domain knowledge.

Let's also look at the **dominant topic** assigned to each document using the `topics()` function.

In [ ]:
# Display the dominant topic for each document
dominant_topics <- topics(lda_model)

cat("Dominant topic per document:\n\n")
for (i in seq_along(dominant_topics)) {
  cat(sprintf("  Doc %2d  %-25s %-18s Topic %d\n",
              i, df$speaker[i], df$party[i], dominant_topics[i]))
}

**Reflection:** Look at the top words for each topic. Can you give each topic a meaningful label? What themes do you see? Remember that with only 20 documents the topics may be less sharply defined than they would be with a larger corpus.

## Visualising Topics

Visualisation is essential for interpreting topic models. We will use the `tidytext` package (Silge and Robinson, 2016) to extract tidy data frames from the fitted model, and `ggplot2` to create publication-quality plots.

### Topic-Word Distributions (Beta)

The **beta** matrix contains the per-topic-per-word probabilities. For each topic, beta tells us the probability of each word being generated by that topic. We can extract this using `tidy(model, matrix = "beta")` from the `tidytext` package.

Let's visualise the top 10 words per topic as a faceted bar chart.

In [ ]:
# Extract the beta (topic-word) matrix in tidy format
topic_word_probs <- tidy(lda_model, matrix = "beta")

cat("Beta matrix (first 10 rows):\n")
head(topic_word_probs, 10)

In [ ]:
# Get the top 10 words per topic
top_terms_plot <- topic_word_probs |>
  group_by(topic) |>
  slice_max(beta, n = 10) |>
  ungroup() |>
  arrange(topic, -beta)

# Create a faceted bar chart
ggplot(top_terms_plot, aes(x = reorder_within(term, beta, topic), y = beta, fill = factor(topic))) +
  geom_col(show.legend = FALSE) +
  facet_wrap(~ topic, scales = "free_y", labeller = labeller(topic = function(x) paste("Topic", x))) +
  coord_flip() +
  scale_x_reordered() +
  labs(
    title = "Top 10 Words per Topic",
    x = NULL,
    y = "Beta (word probability within topic)"
  ) +
  theme_minimal() +
  theme(strip.text = element_text(size = 12, face = "bold"))

The `reorder_within()` and `scale_x_reordered()` functions from `tidytext` ensure that the words are ordered correctly within each facet. Without these, words would be ordered globally rather than per-topic, producing a confusing plot.

The bar lengths represent the probability (beta) of each word within the topic. Words at the top of each panel are the most characteristic of that topic. Compare the topics: do they capture distinct themes in the speeches?

## Document-Topic Assignments

A key output of LDA is the **document-topic distribution** — for each document, the model estimates the proportion of the document that belongs to each topic. This is what makes LDA a "mixed membership" model: a single document can be about multiple topics in different proportions.

We can extract these proportions using `tidy(model, matrix = "gamma")` from the `tidytext` package, or equivalently using `posterior(model)$topics` from the `topicmodels` package.

### Using the Gamma Matrix

The **gamma** matrix contains the per-document-per-topic proportions. For each document, gamma tells us what fraction of the document is assigned to each topic.

In [ ]:
# Extract the gamma (document-topic) matrix in tidy format
doc_topic_probs <- tidy(lda_model, matrix = "gamma")

cat("Gamma matrix (first 12 rows):\n")
head(doc_topic_probs, 12)

### Dominant Topic per Document

Let's identify the dominant topic for each document along with the gamma proportion, which tells us how confident the model is in the assignment.

In [ ]:
# Get the posterior topic distributions
posterior_topics <- posterior(lda_model)$topics

cat(sprintf("%-4s  %-25s %-18s %14s  %10s\n",
            "Doc", "Speaker", "Party", "Dominant Topic", "Proportion"))
cat(paste0(rep("-", 80), collapse = ""), "\n")

for (i in 1:nrow(posterior_topics)) {
  dominant <- which.max(posterior_topics[i, ])
  proportion <- max(posterior_topics[i, ])
  cat(sprintf("%4d  %-25s %-18s %8d        %10.2f\n",
              i, df$speaker[i], df$party[i], dominant, proportion))
}

The **proportion** column tells us how confident the model is in the assignment. A value close to 1.0 means the document is almost entirely about one topic. A value closer to 0.33 (with K=3) means the document is a mixture of all three topics in roughly equal proportions.

Let's look at the full topic distribution for a few documents to see the mixture more clearly.

In [ ]:
# Show full topic distributions for the first 5 documents
cat("Full topic distributions for the first 5 documents:\n\n")

for (i in 1:5) {
  probs <- posterior_topics[i, ]
  dist_str <- paste(sapply(seq_along(probs), function(t) {
    sprintf("Topic %d: %.3f", t, probs[t])
  }), collapse = "  ")
  cat(paste0("Doc ", i, " (", df$speaker[i], "):\n"))
  cat(paste0("  ", dist_str, "\n\n"))
}

This mixed-membership property is one of the strengths of LDA over hard clustering methods. It reflects the reality that parliamentary speeches often touch on multiple themes — a speech about education might also mention the economy or public spending.

### Visualising Document-Topic Distributions

A heatmap can provide a useful overview of how topics are distributed across all documents in the corpus. We use `ggplot2` with `geom_tile()` to create this.

In [ ]:
# Build a document-topic data frame for plotting
doc_topic_df <- as.data.frame(posterior_topics)
colnames(doc_topic_df) <- paste("Topic", 1:ncol(doc_topic_df))
doc_topic_df$document <- paste0(df$speaker, " (", df$party, ")")

# Reshape to long format for ggplot
doc_topic_long <- melt(doc_topic_df, id.vars = "document",
                       variable.name = "topic", value.name = "proportion")

# Preserve document order
doc_topic_long$document <- factor(doc_topic_long$document,
                                   levels = rev(doc_topic_df$document))

# Plot the heatmap
ggplot(doc_topic_long, aes(x = topic, y = document, fill = proportion)) +
  geom_tile(colour = "white") +
  scale_fill_gradient(low = "white", high = "steelblue", name = "Proportion") +
  labs(
    title = "Document-Topic Distributions",
    x = "Topic",
    y = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text.y = element_text(size = 7),
    plot.title = element_text(face = "bold", size = 14)
  )

In the heatmap, darker cells indicate a higher proportion of the document assigned to that topic. Look for patterns: do speakers from the same party tend to cluster on the same topics? Are some topics more dominant across the corpus than others?

## Experimenting with K

So far we have fitted a model with K=3 topics. But how do we know 3 is the right number? In practice, **there is no single correct value of K**. The choice involves both statistical guidance and substantive judgement.

One commonly used statistical measure in the `topicmodels` package is **perplexity**. Perplexity measures how well the model predicts unseen words — it captures how "surprised" the model is by the data. Lower perplexity suggests a better fit to the data.

However, perplexity has an important limitation: it tends to decrease as K increases, because more topics allow the model to fit the data more closely. A model with as many topics as documents would have very low perplexity but would be useless for interpretation. Perplexity is therefore best used as **one input** alongside qualitative assessment of topic interpretability.

Let's fit models with K=2, 3, 4, and 5, and compute the perplexity for each.

In [ ]:
# Fit models with different values of K and compute perplexity
k_values <- c(2, 3, 4, 5)

cat("Fitting models and computing perplexity...\n\n")

perplexities <- sapply(k_values, function(k) {
  m <- LDA(dtm, k = k, control = list(seed = 42))
  p <- perplexity(m)
  cat(sprintf("  K=%d: Perplexity = %.2f\n", k, p))
  return(p)
})

cat("\nDone.")

In [ ]:
# Plot perplexity against K
perp_df <- data.frame(K = k_values, Perplexity = perplexities)

ggplot(perp_df, aes(x = K, y = Perplexity)) +
  geom_line(linewidth = 1, colour = "steelblue") +
  geom_point(size = 3, colour = "steelblue") +
  scale_x_continuous(breaks = k_values) +
  labs(
    title = "Perplexity by Number of Topics",
    x = "Number of Topics (K)",
    y = "Perplexity"
  ) +
  theme_minimal() +
  theme(plot.title = element_text(face = "bold", size = 14))

### Interpreting the Perplexity Plot

The perplexity plot helps guide the choice of K, but it should not be treated as a definitive answer. Some things to keep in mind:

- **Lower perplexity suggests better fit**, but this is only one measure. Perplexity captures statistical fit, not interpretability.
- **Look for an "elbow"** — a point where perplexity decreases sharply and then levels off. This often suggests a reasonable number of topics.
- **Substantive judgement matters.** Even if perplexity is lowest at K=5, it may be more informative to use K=3 if the topics are more coherent and interpretable.
- **With a small corpus** (like our 20 speeches), perplexity values are less reliable. Do not over-interpret small differences.

As Grimmer, Roberts, and Stewart (2022) emphasise: "There is no right answer to the number of topics... the choice should be guided by the research question, the data, and the analyst's interpretation of the topics."

## The Effect of Preprocessing

Preprocessing choices — which words to keep, whether to stem or lemmatise, how aggressively to filter — **materially affect topic model outputs**. Two researchers analysing the same corpus with different preprocessing pipelines may arrive at different topics.

To illustrate this, let's fit a topic model on the same corpus but **without stopword removal** and compare the resulting topics to our original model.

In [ ]:
# Build a DFM WITHOUT stopword removal
dfm_no_stop <- corp |>
  tokens(remove_punct = TRUE, remove_numbers = TRUE) |>
  tokens_tolower() |>
  dfm()

# Apply the same trimming
dfm_no_stop_trimmed <- dfm_trim(dfm_no_stop, min_docfreq = 2, max_docfreq = 0.8, docfreq_type = "prop")

cat(paste0("Vocabulary with stopword removal:    ", ncol(dfm_trimmed), " terms\n"))
cat(paste0("Vocabulary without stopword removal:  ", ncol(dfm_no_stop_trimmed), " terms\n"))

In [ ]:
# Convert and fit LDA without stopword removal
dtm_no_stop <- convert(dfm_no_stop_trimmed, to = "topicmodels")
lda_no_stop <- LDA(dtm_no_stop, k = 3, control = list(seed = 42))

# Compare topics side by side
cat(paste0(rep("=", 70), collapse = ""), "\n")
cat("TOPICS WITH STOPWORD REMOVAL (our original model):\n")
cat(paste0(rep("=", 70), collapse = ""), "\n")
original_terms <- terms(lda_model, 8)
for (j in 1:ncol(original_terms)) {
  cat(paste0("  Topic ", j, ": ", paste(original_terms[, j], collapse = ", "), "\n"))
}

cat("\n")
cat(paste0(rep("=", 70), collapse = ""), "\n")
cat("TOPICS WITHOUT STOPWORD REMOVAL:\n")
cat(paste0(rep("=", 70), collapse = ""), "\n")
no_stop_terms <- terms(lda_no_stop, 8)
for (j in 1:ncol(no_stop_terms)) {
  cat(paste0("  Topic ", j, ": ", paste(no_stop_terms[, j], collapse = ", "), "\n"))
}

### What Do You Notice?

Compare the two sets of topics. Without stopword removal, common words like "the", "and", "of" may dominate the topics (depending on the filtering thresholds), making them harder to interpret. The substantive content words are pushed down the list, and the topics become less distinct.

This demonstrates an important principle: **preprocessing choices materially affect topic model outputs**. The decisions you make about tokenisation, stopword removal, stemming, and filtering are not merely technical — they shape the substantive findings of your analysis.

In practice, it is good research practice to:
1. Report your preprocessing choices clearly.
2. Consider running the model with different preprocessing settings as a **robustness check**.
3. Be transparent about the fact that different choices may yield different results.

## Exercise

Now it is your turn. Your task is to:

1. **Fit a topic model with a different number of topics (K)**. Choose a value of K that we have not yet examined in detail (for example, K=4 or K=5).
2. **Compare the results to our K=3 model:**
   - (a) Are the topics more or less interpretable? Examine the top words to support your answer.
   - (b) Do you see any new themes emerge that were not visible with K=3?
   - (c) Which K do you prefer and why?

Use the skeleton code below to guide you. Replace the `# YOUR CODE HERE` comments with your own code.

In [ ]:
# Step 1: Choose your K and fit the model
my_k <- ___  # Replace ___ with your chosen number of topics

# YOUR CODE HERE: fit an LDA model with your chosen K
# Hint: use the same dtm as before
my_model <- LDA(
  # YOUR CODE HERE
)

In [ ]:
# Step 2: View the topics
# YOUR CODE HERE: print the top 10 terms from your model

In [ ]:
# Step 3: Compute the perplexity
# YOUR CODE HERE: use perplexity() on your model and compare to the K=3 model

**Reflection:** Compare your model to the K=3 model. Answer the three questions above.

*YOUR ANSWER HERE*

## Appendix: Exercise Solution

In [ ]:
# --- Exercise Solution ---

# Step 1: Choose K and fit the model
my_k <- 4

my_model <- LDA(dtm, k = my_k, control = list(seed = 42))

cat(paste0("Fitted LDA model with K=", my_k, " topics.\n"))

In [ ]:
# Step 2: View the topics
cat(paste0("Topics for K=", my_k, ":\n\n"))

my_terms <- terms(my_model, 10)
print(my_terms)

In [ ]:
# Step 3: Compute the perplexity and compare
my_perplexity <- perplexity(my_model)
original_perplexity <- perplexity(lda_model)

cat(sprintf("Perplexity for K=3: %.2f\n", original_perplexity))
cat(sprintf("Perplexity for K=%d: %.2f\n", my_k, my_perplexity))

if (my_perplexity < original_perplexity) {
  cat(sprintf("\nThe K=%d model has lower perplexity (better statistical fit).\n", my_k))
} else {
  cat("\nThe K=3 model has lower perplexity (better statistical fit).\n")
}

**Reflection:** With K=4, the model may split one of the K=3 topics into two more specific sub-themes, or it may produce a new topic that captures a theme not visible at K=3. Whether this is an improvement depends on whether the new topics are substantively meaningful and interpretable. With our small corpus of 20 speeches, increasing K beyond 3 or 4 risks producing topics that are difficult to interpret — there simply is not enough data to support many distinct themes. In practice, the best K is the one that produces topics you can interpret, label, and use to answer your research question.

## Bibliography

- Benoit, K., Watanabe, K., Wang, H., Noel, P., Lori, S., Obeng, A., Muller, S., & Matsuo, A. (2018). quanteda: An R package for the quantitative analysis of textual data. *Journal of Open Source Software*, 3(30), 774.
- Blei, D. M., Ng, A. Y., & Jordan, M. I. (2003). Latent Dirichlet Allocation. *Journal of Machine Learning Research*, 3, 993-1022.
- Grimmer, J., Roberts, M. E., & Stewart, B. M. (2022). *Text as Data: A New Framework for Machine Learning and the Social Sciences*. Princeton University Press.
- Grun, B., & Hornik, K. (2011). topicmodels: An R Package for Fitting Topic Models. *Journal of Statistical Software*, 40(13), 1-30.
- Roberts, M. E., Stewart, B. M., & Tingley, D. (2019). stm: An R Package for Structural Topic Models. *Journal of Statistical Software*, 91(2), 1-40.
- Silge, J., & Robinson, D. (2016). tidytext: Text Mining and Analysis Using Tidy Data Principles in R. *Journal of Open Source Software*, 1(3), 37.

---

**END OF FILE**